<h2>Upload video to youtube</h2>

In [17]:
import os
import pickle

from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

In [ ]:
SCOPES = [
    "https://www.googleapis.com/auth/youtube.upload",   #upload file
    "https://www.googleapis.com/auth/youtube.force-ssl" #create comment
]


In [ ]:
VIDEO_FOLDER = '/Users/sangdo/Downloads/math_games_video/output/'   #contain mp4 files

In [ ]:
YT_CREDENTIAL_FILEPATH = "secret_files/martin_yt_client_secret.json" #download from Google console app
YT_SECRET_FILEPATH = 'secret_file/martin_yt_token.json' #appear after authentication in web

In [ ]:
TITLE_PREFIX = 'Math games for your kids at the spare time - Puzzle '

In [ ]:
#authorize to save permanent token
def get_authenticated_service():
    creds = None
    # load saved token
    if os.path.exists(YT_SECRET_FILEPATH):
        creds = Credentials.from_authorized_user_file(YT_SECRET_FILEPATH, SCOPES)

    # if no valid credentials
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                YT_CREDENTIAL_FILEPATH,
                SCOPES
            )
            creds = flow.run_local_server(port=0)
        # save token
        with open(YT_SECRET_FILEPATH, "w") as token:
            token.write(creds.to_json())

    youtube = build("youtube", "v3", credentials=creds)
    return youtube

# youtube = get_authenticated_service() #permanent token is saved. Note: login with developer account (Tester type)

In [ ]:
def upload_video(file_path, title, description):
    request_body = {
        "snippet": {
            "title": title,
            "description": description
        },
        "status": {
            "privacyStatus": "public"
        }
    }

    media = MediaFileUpload(file_path, resumable=True)

    request = youtube.videos().insert(
        part="snippet,status",
        body=request_body,
        media_body=media
    )

    response = None

    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f"Uploading {int(status.progress() * 100)}%")

    print("Upload complete")
    print("Video ID:", response["id"])

#test
file_path = VIDEO_FOLDER + '12.mp4'
title = TITLE_PREFIX + '12'
description = """
Download more than 300 Math Games as a printable PDF file for your kids here: 

https://sangdomartin.gumroad.com/l/mathgames

We introduce a range of various games:
Addition matrix
Hidden gems
Word search
Crossword numbers
Balance game
Find lines
Triangle sum
Balance fruit
Object coordination
Spy game
Detect shape
Bee house
"""

# upload_video(file_path, title, description)

In [ ]:
def add_comment(video_id, comment_text):
    request = youtube.commentThreads().insert(
        part="snippet",
        body={
            "snippet": {
                "videoId": video_id,
                "topLevelComment": {
                    "snippet": {
                        "textOriginal": comment_text
                    }
                }
            }
        }
    )

    response = request.execute()
    print("Comment posted")
    return response
#
video_id = 'UPJLn_U6hD0'
comment = 'Download more than 300 Math Games as a printable PDF file for your kids here: https://sangdomartin.gumroad.com/l/mathgames'
# add_comment(video_id, comment)    #need phone number to clickable link & wait around 10 minutes to post